# FactorFolio Step 3: Known-Answer Tests for core/models.py

**AC420 project, Jasper Lin.** Before the app uses the regression code in `core/models.py`, this notebook checks that it reproduces answers from class:

* **Tests 1 and 2:** the Lecture 12 & 13 notebook's CAPM and Fama-French regressions for MSFT and XOM (2016 to 2020).
* **Test 3:** the Lecture 17 study guide's MSFT result after the year-month merge fix.
* **Test 4:** portfolio checks: the portfolio's beta equals the weighted average of its stocks' betas, and R-squared equals the systematic share of risk.

**To run in Colab:** upload `Lect-12-output.csv` and `FF-Factors-2016-2020.csv` (Lecture 12 folder) and `stock_returns.csv` and `FF_factors.csv` (Lecture 17 folder), then Runtime, then Run all. The class data stays in Colab; it is never added to the GitHub repo.

Written with AI assistance (Claude); see my AI usage log.

### Get the app's code from GitHub and import the models

In [ ]:
!git clone -q https://github.com/jasperl12/ac420-project-factorfolio-portfolio-risk-app.git factorfolio
import sys
sys.path.append('factorfolio')

import pandas as pd
from core import models

DATA_DIR = '/content/'

### Tests 1 and 2: Lecture 12 & 13 notebook (MSFT and XOM, 60 months)
Same steps as the class notebook: split the stacked file by ticker, rename `dateff` to `date`, merge, and build the excess return `RET - rf`.

In [ ]:
data = pd.read_csv(DATA_DIR + 'Lect-12-output.csv')
ff_factors = pd.read_csv(DATA_DIR + 'FF-Factors-2016-2020.csv')
ff_factors.rename(columns={'dateff': 'date'}, inplace=True)

results = []
for tic in ['MSFT', 'XOM']:
    stock_ff = pd.merge(data[data['TICKER'] == tic], ff_factors, on='date', how='inner')
    print(tic, 'rows after merge:', len(stock_ff))       # should be 60
    y = stock_ff['RET'] - stock_ff['rf']
    capm = models.run_capm(y, stock_ff['mktrf'])
    ff3 = models.run_ff3(y, stock_ff)
    results.append({'ticker': tic, 'capm_alpha': capm['alpha'], 'capm_beta': capm['beta'], 'capm_r2': capm['r2'],
                    'ff3_alpha': ff3['alpha'], 'ff3_beta': ff3['beta'], 'ff3_s': ff3['s'], 'ff3_h': ff3['h'],
                    'ff3_r2': ff3['r2']})
lecture12 = pd.DataFrame(results).set_index('ticker')
lecture12.round(4)

In [ ]:
# Answers from the Lecture 12 and 14 study guides
expected = pd.DataFrame({
    'capm_alpha': [0.0157, -0.0211], 'capm_beta': [0.756, 1.293], 'capm_r2': [0.452, 0.616],
    'ff3_alpha': [0.0114, -0.0149], 'ff3_beta': [0.938, 1.092], 'ff3_s': [-0.498, 0.429],
    'ff3_h': [-0.349, 0.554], 'ff3_r2': [0.560, 0.695]}, index=['MSFT', 'XOM'])
gap = (lecture12 - expected).abs()
print(gap.round(4))
assert (gap < 0.001).all().all(), 'A Lecture 12 answer does not match'
print('Tests 1 and 2 passed: every estimate matches the class answers to 3 decimals.')

### Test 3: Lecture 17 study guide (MSFT after the year-month merge fix)
Expected: 64 months, CAPM beta 1.064, FF3 R-squared 0.614.

In [ ]:
factors = pd.read_csv(DATA_DIR + 'FF_factors.csv').rename(columns={'dateff': 'datadate'})
returns = pd.read_csv(DATA_DIR + 'stock_returns.csv', usecols=['datadate', 'tic', 'trt1m'])
returns = returns.dropna(subset=['trt1m'])
returns['ret'] = returns['trt1m'] / 100
returns['ym'] = pd.to_datetime(returns['datadate']).dt.to_period('M')
factors['ym'] = pd.to_datetime(factors['datadate']).dt.to_period('M')
combined = pd.merge(returns, factors.drop(columns='datadate'), on='ym', how='inner')

msft = combined[combined['tic'] == 'MSFT']
y = msft['ret'] - msft['rf']
capm = models.run_capm(y, msft['mktrf'])
ff3 = models.run_ff3(y, msft)
print('months:', capm['n_months'], '| CAPM beta:', round(capm['beta'], 3), '| FF3 R2:', round(ff3['r2'], 3))
print(models.alpha_label(capm))
assert capm['n_months'] == 64 and abs(capm['beta'] - 1.064) < 0.001 and abs(ff3['r2'] - 0.614) < 0.001
print('Test 3 passed.')

### Test 4: Portfolio checks (sample portfolio from the app)
* The portfolio's monthly return is the weighted sum of its stocks' returns.
* Over the same months, the portfolio's beta, size and value loadings equal the weighted averages of its stocks' loadings (the suggestions engine relies on this).
* R-squared equals the systematic share of risk, and diversification should make the portfolio's R-squared higher than the typical stock's.

In [ ]:
weights = {'AAPL': 30, 'MSFT': 20, 'NVDA': 20, 'KO': 15, 'JNJ': 15}
wide = combined.pivot_table(index='ym', columns='tic', values='ret')[list(weights)]
window = wide.join(factors.set_index('ym')[['mktrf', 'smb', 'hml', 'rf']], how='inner').dropna()
ds = {'window': window, 'tickers': list(weights)}

result = models.analyze_portfolio(ds, weights, benchmark=None)
stocks, port = result['stocks'], result['portfolio']
print(stocks[['weight', 'beta', 's', 'h', 'r2', 'volatility']].round(3))

w = result['weights']
checks = {'beta': (stocks['beta'] * w).sum(), 'ff3_beta': (stocks['ff3_beta'] * w).sum(),
          's': (stocks['s'] * w).sum(), 'h': (stocks['h'] * w).sum()}
for k, weighted_avg in checks.items():
    print(f'portfolio {k}: {port[k]:.4f}  weighted average of stocks: {weighted_avg:.4f}')
    assert abs(port[k] - weighted_avg) < 1e-10
assert abs(port['split']['systematic_share'] - port['r2']) < 1e-10
print('portfolio R2:', round(port['r2'], 3), '| average stock R2:', round(stocks['r2'].mean(), 3))
assert port['r2'] > stocks['r2'].mean()
print('Test 4 passed.')